# Joystick test (2-axis, via the ADS7830)

**Wiring** (5 pins on the joystick board):
| Joystick pin | Connect to |
|---|---|
| GND | GND |
| +5V (or VCC) | **3.3V** rail, so the readings stay inside the ADC's range |
| VRx | ADS7830 **A2** |
| VRy | ADS7830 **A1** |
| SW | leave unconnected for now |

Run the cell and move the stick. At rest both numbers should sit near 128. Pushing all the way should take a number close to 0 one way and close to 255 the other.

Make a note of which way `y` goes when you push **up**. We'll need that when wiring it into the motor code. If "up" and "down" come out swapped in the text, that's fine, it's just a setting to flip later.

In [ ]:
import time
from smbus2 import SMBus

ADS7830_ADDR = 0x4b
JOY_X_CHANNEL = 2       # joystick VRx on A2
JOY_Y_CHANNEL = 1       # joystick VRy on A1
CENTER = 128            # expected resting value (0..255)
DEADBAND = 40           # how far from centre before it counts as a push


def read_adc(bus, channel):
    """Return 0..255 from an ADS7830 channel."""
    cmd = 0x84 | ((((channel << 2) | (channel >> 1)) & 0x07) << 4)
    return bus.read_byte_data(ADS7830_ADDR, cmd)


def read_joystick(bus):
    """Return (x, y) as raw 0..255 values."""
    return read_adc(bus, JOY_X_CHANNEL), read_adc(bus, JOY_Y_CHANNEL)


def describe(x, y):
    """Turn raw readings into words like 'up', 'down-left', or 'centre'."""
    parts = []
    if y < CENTER - DEADBAND:
        parts.append("up")
    elif y > CENTER + DEADBAND:
        parts.append("down")
    if x < CENTER - DEADBAND:
        parts.append("left")
    elif x > CENTER + DEADBAND:
        parts.append("right")
    return "-".join(parts) if parts else "centre"


def show_joystick(interval=0.25):
    with SMBus(1) as bus:
        print("Move the joystick. Press stop to end.")
        try:
            while True:
                x, y = read_joystick(bus)
                print(f"x={x:3d}   y={y:3d}   {describe(x, y)}")
                time.sleep(interval)
        except KeyboardInterrupt:
            print("Stopped.")


show_joystick()

## Troubleshooting
- **Both numbers stuck at 0**: the joystick has no power, or VRx/VRy aren't reaching A2/A1.
- **A number stuck at 255 and never moves**: that axis is wired to 5V instead of 3.3V, or the wire is on the wrong pin.
- **Resting value far from 128**: normal for cheap joysticks. Set `CENTER` to whatever it rests at.
- **"up" and "down" swapped**: just a matter of how the stick is oriented. Swap the comparisons in `describe`, or we'll flip it in the motor code.